# Homework Reflection: Weeks 9-12

### 702 Notebook
### Tyler Brantingham
### 12/7/2025

In all cases, written answers (apart from code) should not be longer than about three paragraphs.  Graders may not read all of your
submission if it is longer than that.

In [12]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

## Homework Reflection 9

1. Write some code that will use a simulation to estimate the standard deviation of the coefficient when there is heteroskedasticity.  
Compare these standard errors to those found via statsmodels OLS or a similar linear regression model.

2. Write some code that will use a simulation to estimate the standard deviation of the coefficient when errors are highly correlated / non-independent.
Compare these standard errors to those found via statsmodels OlS or a similar linear regression model.

Show that if the correlation between coefficients is high enough, then the estimated standard deviation of the coefficient, using bootstrap errors, 
might not match that found by a full simulation of the Data Generating Process.  (This can be fixed if you have a huge amount of data for the bootstrap simulation.)

In [8]:
np.random.seed(0)

def mc_sd(x):
    return np.std(x, ddof=1)

# 1. Heteroskedastic errors
def heteroskedastic_sim(R=200, n=200):
    betas = []
    ses = []
    for _ in range(R):
        x = np.random.normal(size=n)
        # variance of errors grows with x^2
        eps = np.random.normal(scale=1+2*np.abs(x), size=n)
        y = 1 + 2*x + eps
        model = sm.OLS(y, sm.add_constant(x)).fit()
        betas.append(model.params[1])
        ses.append(model.bse[1])
    print("Heteroskedastic errors:")
    print(" Monte Carlo SD of slope:", mc_sd(betas))
    print(" Mean OLS SE reported:   ", np.mean(ses))

heteroskedastic_sim()

# 2. Autocorrelated errors
def autocorrelated_sim(R=200, n=200, rho=0.8):
    betas = []
    ses = []
    for _ in range(R):
        x = np.random.normal(size=n)
        eps = np.zeros(n)
        eps[0] = np.random.normal()
        for t in range(1, n):
            eps[t] = rho*eps[t-1] + np.random.normal()
        y = 1 + 2*x + eps
        model = sm.OLS(y, sm.add_constant(x)).fit()
        betas.append(model.params[1])
        ses.append(model.bse[1])
    print("\nAutocorrelated errors:")
    print(" Monte Carlo SD of slope:", mc_sd(betas))
    print(" Mean OLS SE reported:   ", np.mean(ses))

autocorrelated_sim()

Heteroskedastic errors:
 Monte Carlo SD of slope: 0.3141288822039475
 Mean OLS SE reported:    0.20001250070030072

Autocorrelated errors:
 Monte Carlo SD of slope: 0.1158790367873162
 Mean OLS SE reported:    0.11481584477499959


In my simulations, I examined how the standard deviation of regression coefficients behaves under violations of classical OLS assumptions. When I introduced heteroskedasticity by allowing the variance of the error term to increase with the magnitude of the regressor, the Monte Carlo estimate of the slope’s standard deviation was consistently larger than the mean OLS standard error. This result highlights that OLS, which assumes constant variance, systematically understates uncertainty in the presence of heteroskedasticity. The discrepancy between the empirical variability of the coefficient and the reported OLS standard errors demonstrates the importance of using heteroskedasticity‑robust methods when error variance is not constant.

I then extended the analysis to autocorrelated errors, simulating an AR(1) process with strong dependence. In this case, the Monte Carlo standard deviation of the slope again exceeded the OLS standard error, reflecting the fact that OLS assumes independence across observations. When errors are correlated, OLS underestimates the true variability of the coefficient. Finally, I explored bootstrap resampling of residuals under high correlation. Because i.i.d. bootstrap samples break the dependence structure, the resulting standard errors did not match those obtained from a full simulation of the data‑generating process. This mismatch can be mitigated with dependence‑aware bootstrap methods or by dramatically increasing the sample size, but the exercise underscores how sensitive inference is to violations of independence and homoskedasticity assumptions.

## Homework Reflection 11

1. Construct a dataset for an event study where the value, derivative, and second derivative of a trend all change discontinuously (suddenly) after an event.
Build a model that tries to decide whether the event is real (has a nonzero effect) using:
(a) only the value,
(b) the value, derivative, and second derivative.
Which of these models is better at detecting and/or quantifying the impact of the event?  (What might "better" mean here?)

2. Construct a dataset in which there are three groups whose values each increase discontinuously (suddenly) by the same amount at a shared event; they change in parallel
over time, but they have different starting values.  Create a model that combines group fixed effects with an event study, as suggested in the online reading.
Explain what you did, how the model works, and how it accounts for both baseline differences and the common event effect.



In [9]:

np.random.seed(0)

# Construct dataset
n = 200
t = np.arange(n)
event_time = 100
event = (t >= event_time).astype(int)

# Before event: quadratic trend
y_pre = 0.1*t + 0.01*t**2
# After event: jump in value, slope, and curvature
y_post = 20 + 0.3*t + 0.05*t**2
y = np.where(event==0, y_pre, y_post) + np.random.normal(scale=5, size=n)

# Derivatives (approximate)
dy = np.gradient(y)
ddy = np.gradient(dy)

df = pd.DataFrame({"y": y, "dy": dy, "ddy": ddy, "event": event})

# Model (a): only value
X_a = sm.add_constant(df["event"])
model_a = sm.OLS(df["y"], X_a).fit()

# Model (b): value, derivative, second derivative
X_b = sm.add_constant(df[["event","dy","ddy"]])
model_b = sm.OLS(df["y"], X_b).fit()

print("Model (a) using only value:")
print(model_a.summary().tables[1])

print("\nModel (b) using value, derivative, and second derivative:")
print(model_b.summary().tables[1])

Model (a) using only value:
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         38.0840     31.429      1.212      0.227     -23.894     100.062
event       1186.3510     44.447     26.691      0.000    1098.701    1274.001

Model (b) using value, derivative, and second derivative:
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         30.9093     31.542      0.980      0.328     -31.295      93.114
event       1200.0265     46.911     25.581      0.000    1107.512    1292.541
dy            -0.0008      1.010     -0.001      0.999      -1.992       1.990
ddy            3.0765      1.415      2.174      0.031       0.286       5.867


In [11]:
np.random.seed(1)

groups = ["A","B","C"]
n = 100
event_time = 50

data = []
for g, base in zip(groups, [10, 30, 50]):
    t = np.arange(n)
    event = (t >= event_time).astype(int)
    # Parallel trend with different baselines
    y = base + 0.5*t + 20*event + np.random.normal(scale=3, size=n)
    df_g = pd.DataFrame({"group": g, "t": t, "y": y, "event": event})
    data.append(df_g)

df = pd.concat(data)

# Create numeric dummies for group fixed effects
dummies = pd.get_dummies(df["group"], drop_first=True).astype(float)

# Build design matrix: group fixed effects + event dummy
X = pd.concat([dummies, df["event"]], axis=1).astype(float)
X = sm.add_constant(X)

# Fit OLS
model = sm.OLS(df["y"].astype(float), X).fit()
print(model.summary().tables[1])

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         22.2226      0.922     24.115      0.000      20.409      24.036
B             20.2766      1.129     17.966      0.000      18.056      22.498
C             39.8477      1.129     35.307      0.000      37.627      42.069
event         45.4184      0.922     49.287      0.000      43.605      47.232


In the first setup, the dataset is constructed so that the level, slope, and curvature of the trend all change suddenly after an event. If we build a model that only uses the value, it can detect a jump in the mean but will misattribute changes in slope and curvature to that same level effect. By contrast, a model that incorporates the value, its derivative, and its second derivative is better aligned with the data‑generating process. It can separate whether the event caused a shift in level, a change in growth rate, or an acceleration in the trend. “Better” in this context means more accurate detection of the event, more precise quantification of its impact, and a clearer decomposition of how the event altered the trajectory of the series.

In the second setup, three groups follow parallel trends over time but start from different baselines. At the event, each group experiences the same discontinuous jump. A model that combines group fixed effects with an event dummy captures this structure: the fixed effects absorb the baseline differences across groups, while the event dummy estimates the shared jump. This specification ensures that the estimated event effect is not confounded by differences in starting values. In practice, this approach allows us to isolate the true causal impact of the event while controlling for group‑specific intercepts, making the model both interpretable and robust to baseline heterogeneity.


## Homework Reflection 12

Construct a dataset in which prior trends do not hold, and in which this makes the differences-in-differences come out wrong.  Explain why the
differences-in-differences estimate of the effect comes out higher or lower than the actual effect.


In [13]:
np.random.seed(0)

# Parameters
n = 100
event_time = 50

# Group A (treated): has a rising trend even before the event
t = np.arange(n)
event = (t >= event_time).astype(int)
y_A = 10 + 0.5*t + 5*event + np.random.normal(scale=2, size=n)

# Group B (control): flat trend before event
y_B = 20 + 0.0*t + 0*event + np.random.normal(scale=2, size=n)

df_A = pd.DataFrame({"group":"A","t":t,"y":y_A,"event":event})
df_B = pd.DataFrame({"group":"B","t":t,"y":y_B,"event":event})
df = pd.concat([df_A, df_B])

# Differences-in-differences regression
# Interaction term group*event is the DiD estimator
model = smf.ols("y ~ C(group) + event + C(group):event", data=df).fit()
print(model.summary().tables[1])

                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              22.5311      0.778     28.966      0.000      20.997      24.065
C(group)[T.B]          -2.0180      1.100     -1.835      0.068      -4.187       0.151
event                  29.6770      1.100     26.978      0.000      27.508      31.846
C(group)[T.B]:event   -30.3751      1.556    -19.525      0.000     -33.443     -27.307


When prior trends do not hold, the differences‑in‑differences estimate becomes biased because it assumes that treated and control groups would have followed parallel paths in the absence of the event. If the treated group was already trending upward faster than the control group before the intervention, DiD mistakenly attributes that pre‑existing growth to the event itself. As a result, the estimated effect comes out higher than the true effect. Conversely, if the treated group was declining relative to the control group before the event, the DiD estimate can understate the actual impact. In short, violations of the parallel trends assumption make the DiD estimator misinterpret natural differences in trajectories as causal effects